In [ ]:
Author: Hui Fang

Purpose: ST 554 Project 2

Date: 3/15/2026

# Part I - Creating a Class

We are going to create our own class called SparkDataCheck that works on Spark SQL style data frames.

Create a .py file.

First import modules needed:

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import pyspark.pandas as ps

## Create a class Called SparkDataCheck

- Start a class called **SparkDataCheck**
- Create an `__init__` function that takes in self and a dataframe argument

  – Within this, create a `.df` attribute that is the dataframe

Let's start by creating our spark session

In [19]:
from pyspark.sql import SparkSession         # import the SparkSession class from PySpark#
spark = SparkSession.builder.getOrCreate()   # create or retrieve a SparkSession

Load a sample dataset from testing.

In [6]:
pdf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/red-wine.csv", delimiter = ";")
pdf.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


### Initiate a class and two classmethods

Initiate a class and create two @classmethods:

– One that creates an instance while reading in a csv file.\
   ∗ The method should have arguments for the class, the spark session, and the path to the file\
   ∗ You should use the `spark.read.load()` function as we did in our `pyspark` notebook. \
   ∗ Create an object of our class that is returned 
   
– One that creates an instance from a `pandas` dataframe (standard `pandas`)\
  ∗ The method should have arguments for the class, the spark session, and the pandas dataframe\
  ∗ You should use the `spark.CreateDataFrame()` function as we did in our `pyspark` notebook.\
  ∗ Create an object of our class that is returned

In [1]:
"""
SparkDataCheck.py

This module define a class that works 
on Spark SQL style data frames.
"""
# import modules needed
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql.types import NumericType
from pyspark.sql.functions import col as spark_col

class SparkDataCheck:
    def __init__(self, df: DataFrame):
        # create a .df attribute
        self.df = df
            
  #=================================================================
    # Classmethod 1: create instance by reading a CSV file
    @classmethod   
    def from_csv(cls, spark, path):
        df = (spark.read
                   .format("csv")
                   .option("header", True)
                   .option("inferSchema", True)
                   .option("sep", sep)
                   .load(path))
        return cls(df)
   
  
    #==============================================================
    # Classmethod 2: create instance from a pandas DataFrame
    @classmethod
    def from_pandas(cls,spark, pandas_df):
        df = spark.createDataFrame(pandas_df)
        return cls(df)
    
         
    

### Test the classmethods

In [49]:
import importlib
import ST554_project2_part1
importlib.reload(ST554_project2_part1)

<module 'ST554_project2_part1' from '/home/jupyter-hfang4@ncsu.edu/ST-554-Project-2/ST554_project2_part1.py'>

In [50]:
# check with the red-wine.csv data
check_df = ST554_project2_part1.SparkDataCheck.from_csv(spark, "red-wine.csv")
check_df.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

In [51]:
import pandas as pd
pdf = pd.read_csv("red-wine.csv")
check_df2 = ST554_project2_part1.SparkDataCheck.from_pandas(spark, pdf)
check_df2.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

This shows that the two classmethods are working. 

### Create validation methods

- Create a couple of validation methods. Each validation method will **modify the df attribute of the object** using different column functions and **return itself (the object that
has the df attribute)** so that we can chain commands. We won’t do any of the returning of the data within our methods (such as `take()` or `collect()`). We’ll leave that as something the user can do
themselves! (** This will need to be done via something like `my_object.df.show()`).

#### Create Boolean column based on numeric bounds

– Create a method that checks if each value in a numeric column is within user defined limits (upper and lower bounds, inclusive) and returns the dataframe with an appended column of
Boolean values.

∗ The function should allow the user to supply a single column and a `lower` and `upper` value. Check that at least one of lower or upper is provided (if not provided, don’t check that side).\
∗ For any `NULL` values, return `NULL`\
∗ If the user supplies a non-numeric column (not float, int, longint, bigint, double, or integer), print a message and return the df without modification.\
∗ Hints: Check out the `.dtypes` attribute of the data frame. On a column, you can use the `.between()` method

In [8]:
#============================================
# 1. Validation methods
#============================================

# 1.1 create boolean column based on numeric bounds

def check_numeric_range(self, col: str, lower: float = None, upper: float = None):
    """
    Append a Boolean column indicating whether values in a numeric column
    fall within user-defined lower and/or upper bounds (inclusive).
    NULL values remain NULL.
    Modifies self.df and returns self for method chaining.
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self

    # -----------------------------------
    # check if the columin is numeric
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, NumericType):
        print(f"Column '{col}' is not numeric.")
        return self

    #--------------------------------------
    # ensure at least one bound is provided
    #--------------------------------------
    if lower is None and upper is None:
        print("No bounds provided. Please provide at least one bound.")
        return self

    #----------------------------------
    # build the Boolean condition   
    #----------------------------------
    if lower is not None and upper is not None:
        # use Spark's between() when both bounds exist
        condition = spark_col(f"`{col}`").between(lower, upper)
    elif lower is not None:
        # only lower bound provided
        condition = spark_col(f"`{col}`") >= lower
    elif upper is not None:
        # only upper bound provided
        condition = spark_col(f"`{col}`") <= upper

    #---------------------------------
    # append Boolean column to dataframe
    #---------------------------------
    new_col_name = f"{col}_in_range"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
    

#### Create a method checking string columns

– Create a method that checks if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values.
∗ For any `NULL` values, return `NUL`\
∗ If the user supplies a non-string column print a message and return the df without modification\
∗ Hint: The `.isin()` method on a column is useful!\

In [21]:
# 1.2 create a method checking values fall within a set of levels

def check_value_levels(self, col: str, levels):
    """
    Check whether values in a string column fall within 
    a user-specified set of allowed levels. 
    Appends a Boolean column. NULL values remain NULL.
    Modifies self.df and returns self for method chaining. 
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self
    
    # -----------------------------------
    # check if the columin is string type
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, StringType):
        print(f"Column '{col}' is not a string column.")
        return self
      
    # build the Boolean condition
    condition = spark_col(col).isin(levels)
    
    # append the new Boolean column
    new_col_name = f"{col}_in_levels"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
        
    

#### Create a method checking missiing values

– Create a method that checks if a each value in a column is missing (`NULL` specifically) and returns the dataframe with an appended column of Boolean values.
∗ Hint: The `.isNULL()` method on a column is useful!

In [38]:
# 1.3 create a method that checks if each value in a column is missing

def check_value_missing(self, col: str):
    """
    Check whether values in a given column are NULL.
    Appends a Boolean column indicating NULL status.
    Modifies self.df and returns self for method chaining.
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self
    
    # build the Bollean contion
    # .isNull() returns True, False, or NULL
    condition = self.spark_col(f"`{col}`").isNull()
    
    # append the new Boolean column to the dataframe
    new_col_name = f"{col}_is_null"
    self.df = self.df.withColumn(new_col_name, condition)
    
    return self
    

### Create summarization methods

- Create a couple of summarization methods (this will generally be writing our own way to use functions that exist!). Each summarization method will return the summarizations
of the data (not an amended version of the dataframe) as a `pandas` data frame (regular `pandas` not `pandas-on-spark`).

#### Create a method to report min and max of a variable

– Create a method to report the `min` and `max` of a numeric column supplied by the user. Add an optional grouping variable (only one grouping variable allowed for simplicity).\
  ∗ The method should check if the column is numeric. If so, it should report the min and max of the column (grouped if appropriate). If not, a message should be printed that the column isn’t numeric and `None` should be returned\
  ∗ If no column is supplied, the method should report the `min` and `max` of any numeric columns (and produce no messages otherwise), grouped if appropriate\
  ∗ Hints: This part got a bit complicated but I used the min and max functions from `pyspark.sql.functions` and the `.agg()` method on a `.groupBy()` spark SQL style data frame (similar to the notes). For the grouped option with all numeric columns, I used `reduce()` from `functools` with `pd.merge()` to simplify the result into a single data frame.

In [3]:
#====================================
# 2. Summarization methods
#====================================

# -----------------------------------------------------------------------------------
# 2.1 define a method to report min and max of a numeric coulumn supplied by the user
# -----------------------------------------------------------------------------------
# import module needed
from pyspark.sql.functions import min, max  

def min_max(self, col = None, group = None):
    """
    Report min and max for:
    1. a user-supplied numeric column (grouped if provided)
    2. or all numeric columns if no column is supplied (grouped if provided)
    Returns a pandas DataFrame
    """
    #-------------------------------------
    # scenario 1: User supplies a column
    #-------------------------------------
    if col is not None:
        # check column exists
        if col not in self.df.columns:
            print(f"Column {col} don't exist.")
            return None
        
        # get the data type of the column
        dtype = self.df.schema[col].dataType

        # check if column is numeric
        if not isinstance(dtype, NumericType):
            print(f"Column '{col}' is not numeric.")
            return None
        
        # check group exists if provided
        if group is not None and group not in self.df.columns:
            print(f"Group column '{group}' does not exist.")
            return None
        
        # grouped version for a single numeric column
        if group is not None:
            return(
                self.df.groupBy(group)
                       .agg(self.min(col).alias(f"{col}_min"),
                            self.max(col).alias(f"{col}_max"))
                       .toPandas()
            )
        
        # ungrouped version for a single numeric column      
        return (
            self.df.select(
                self.min(col).alias(f"{col}_min"),
                self.max(col).alias(f"{col}_max")
            ).toPandas()
        )
    
    #----------------------------------------------------------------
    # scenario 2: no column supplied: compute for all numeric columns
    #----------------------------------------------------------------
    # identify all numeric columns
    numeric_cols = [
        field.name
        for field in self.df.schema.fields
        if isinstance(field.dataType, NumericType)]
    
    # if no numeric columns exist, print a message
    if not numeric_cols:
        print("No numeric columns found.")
        return None
    
     # check group exists if provided
    if group is not None and group not in self.df.columns:
        print(f"Group column '{group}' does not exist.")
        return None
    
    # grouped version for all numeric columns
    if group is not None:
        # setup an enpty list
        dfs = []
        # compute grouped min/max for each numeric column 
        for c in numeric_cols:
            df_c = (
                self.df.groupBy(group)
                       .agg(self.min(c).alias(f"{c}_min"),
                            self.max(c).alias(f"{c}_max"))
            ).toPandas()
            dfs.append(df_c)
            
        # merge all grouped results into on DataFrame
        merged = reduce(lambda left, right: pd.merge(left, right, on = group),dfs)
        return merged
    
    # ungrouped scenartion for all numeric columns
    agg_exprs = []
    for c in numeric_cols:
        agg_exprs.extend([
            self.min(c).alisa(f"{c}_min"),
            self.max(c).alisa(f"{c}_max")
        ])
    # Return a DataFrame with all min/max values
    return self.df.select(*agg_exprs).toPandas()

#### Create a method to report counts of string columns

– Create a method to report the counts associated with one or two string columns. Have the function take in two separate arguments for columns, with the second being optional and the first
required.\
    ∗ The method should check if the column(s) are strings. If so, it should report the counts for the combinations of levels of each variable or of the single variable. If not, a message should
be printed that the column is numeric.

In [4]:
# 2.2 create a method that reports counts of one of two string columns
from pyspark.sql.functions import col as spark_col

def count_column(self, col1: str, col2: str = None):
    """
    Report counts associated with one or two string columns.
    The first column is required; the second is optional.
    Only string columns are allowed. If a column is not string,
    a message is printed and no counts are reported.
    """
    # -----------------------------------
    # check if column1 exists
    # -----------------------------------
    if col1 not in self.df.columns:
        print(f"Column '{col1}' does not exist.")
        return self
    
    # -----------------------------------------
    # if column is provided, check if it exists
    # -----------------------------------------
    if col2 is not None and col2 not in self.df.columns:
        print(f"Column '{col2}' does not exist.")
        return self
    
    # -----------------------------------
    # check if columin1 is string
    #------------------------------------
    dtype1 = self.df.schema[col1].dataType
    if not isinstance(dtype1, StringType):
        print(f"Column '{col1}' is not a string column.")
        return self
    
    # ----------------------------------------------
    # if column2 is provided, check if it is string
    #-----------------------------------------------
    if col2 is not None:
        dtype2 = self.df.schema[col2].dataType
        if not isinstance(dtype2, StringType):
            print(f"Column '{col2}' is not a string column.")
            return self
        
    #-----------------------------------
    # one-column case
    #-----------------------------------
    if col2 is None:
        print(f"Counts for '{col1}':")
        self.df.groupBy(col1).count().show()
        return self
    
    #----------------------------------------------
    # two-column case: both col1 and col2 provided
    #---------------------------------------------
    print(f"Counts for columns '{col1}' and '{col2}':")
    self.df.groupBy(col1, col2).count().show()
    
    return self    
    

## Check the created class with real data

Now we’ll use your class on some data! Create a .ipynb on the JupyterHub (use this same file for the steps
below and part II)
- In the notebook, provide an introduction and narrative to what you are about to do!
- Import your script so you have access to your class!
- Read in the air quality data we used in the first project. I’ve downloaded this data as a .csv file
and it is available at https://www4.stat.ncsu.edu/online/datasets/air.csv. Use your method
that creates an instance of the class from this csv file.
- Provide 4-5 examples of using each of your methods on this object. Show some examples where the
messages need to print out, where only one bound is provided, etc.
- Now, read that same data set in using pandas (not pandas-on-spark). Use your method to create an
instance of this class from the pandas data frame.
- Provide 1 example method call on that object.

### Instroduction

I developed a custom Python class, **SparkDataCheck**, which includes two `classmethods` for creating class instances, three validation methods that append Boolean columns to a Spark DataFrame, and two summarization methods that return pandas DataFrames. To demonstrate that the class works correctly on real data, I will use the [Air Quality dataset](https://archive.ics.uci.edu/dataset/360/air+quality) from the UCI Machine Learning Repository.

I begin by importing my Python script and reading the downloaded air quality CSV file. Using my `from_csv` classmethod, I create a `SparkDataCheck` object and apply each of my methods to this dataset. For each method, I provide four to five examples, including cases where the method prints warning messages (e.g., when a column does not exist or is the wrong type).

Next, I read the same dataset using pandas and use my `from_pandas` classmethod to create a second instance of the class. I then demonstrate one method on this pandas‑based object to confirm that both classmethods work as intended.

### Import my python script and clear up data

In [98]:
# Import my python script
from ST554_project2_part1 import SparkDataCheck

from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("my_app").getOrCreate())
# create object
air = SparkDataCheck.from_csv(spark, "air.csv", sep = ",")

Check data schema

In [4]:
air.df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



Replace missing values with NULL, and create a categrical variable for test created method.

After replacing all missing values (coded as –200) with NULL, I created a new categorical string variable to test the validation methods (such as `check_value_levels`). Specifically, I derived a variable called `Ben` from the numeric column `C6H6(GT)` by grouping its values into four categories based on the distribution of the data:
- < 4.5 -> "Low"
- 4.5 - 8 -> "Medium"
- 8 - 14 -> "High"
- 14 -> "Very high"

In [99]:
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F

# replace missing value with None
for c, dtype in air.df.dtypes:
    if dtype in ("int", "double", "float", "bigint"):
        air.df = air.df.withColumn(c, when(col(f"`{c}`") == -200, None).otherwise(col(f"`{c}`")))
        
# create a new categorical vaiable "Ben"
air.df = air.df.withColumn("Ben",
                          when(col("C6H6(GT)") < 4.5, "Low")
                          .when(col("C6H6(GT)") < 8, "Medium")
                          .when(col("C6H6(GT)") < 14, "High")
                          .otherwise("Very high")
                          )

# show the first 10 rows of the dataset
air.df.show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|   Ben|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|  High|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|  High|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|  High|
|  3|3/10/

26/03/17 23:46:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hfang4@ncsu.edu/ST-554-Project-2/air.csv


### Validation methods

#### Method 1: check numeric_range

In [11]:
# example 1
air.check_numeric_range("CO(GT)", 1232, 2654)

This output means that the method executed with no error. It returned `self`, which is my `SparkDataCheck` instance, and the memory address is 0x7f936e9c7850.

In [43]:
# example 2
air.check_numeric_range("CO2", 20, 500)

Column 'CO2' does not exist.


The output means that the method executed without error. It also printed the message `"Column 'CO2' does not exist."` as expected.

In [30]:
# example 3
air.check_numeric_range("Date", 1, 50)

Column 'Date' is not numeric.


The output means that the method executed without error. It also printed the message "Column 'Date' is not numeric." as expected.

In [34]:
# example 4
air.check_numeric_range("NMHC(GT)", 1000)

The output shows that the method executed without error. Because I provided only one numeric value, the method treated it as the lower bound by default and completed successfully.

In [12]:
# example 5
air.check_numeric_range("PT08.S1(CO)", upper = 1000)

The output shows that the method executed without error. Because I provided only the upper numeric value, the method completed successfully.

#### Method 2: check value levels in string columns

In [23]:
# example 1
air.check_value_levels("Music", ["Good"])

Column 'Music' does not exist.


The output shows that the method executed without error. It detected that the column "Music" does not exist and printed out the message as expected.

In [24]:
# example 2
air.check_value_levels("NO2(GT)", "High")

Column 'NO2(GT)' is not a string column.


The output shows that the method executed without error. It detected the input column is not a string type and printed out the message as expected.

In [25]:
# example 3
air.check_value_levels("PT08.S4(NO2)", "Low")

Column 'PT08.S4(NO2)' is not a string column.


The output shows that the method executed without error. It detected the input column is not a string type and printed out the message as expected.

In [26]:
# example 4
air.check_value_levels("Ben", ["Low", "Medium"])
air.df.select("Ben", "Ben_in_levels").show(10)

+------+-------------+
|   Ben|Ben_in_levels|
+------+-------------+
|  High|        false|
|  High|        false|
|  High|        false|
|  High|        false|
|Medium|         true|
|Medium|         true|
|   Low|         true|
|   Low|         true|
|   Low|         true|
|   Low|         true|
+------+-------------+
only showing top 10 rows


The output shows that the method executed without error. The `check_value_levels` method creates a Boolean indicator showing whether each value in the `Ben` column matches one of the user‑specified allowed levels. Because the method performs a literal string comparison, only rows where the `Ben` value exactly equals one of the supplied strings (e.g., `"Low", "Medium"`) are marked `true` in the new column called `Ben_in_levels`. Rows whose values don't match the allowed list are marked `false`.

#### Method 3: check value missing

In [65]:
# example 1
air.check_value_missing("Benzene")

Column 'Benzene' does not exist.


The output shows that the method executed without error. It detected the input column `Benzene` does not exist and printed out the message as expected.

In [102]:
# example 2
air.check_value_missing("Ben")
air.df.select("Ben", "Ben_is_null").show(5)

+------+-----------+
|   Ben|Ben_is_null|
+------+-----------+
|  High|      false|
|  High|      false|
|  High|      false|
|  High|      false|
|Medium|      false|
+------+-----------+
only showing top 5 rows


The output indicates that the method executed successfully. The `check_value_missing` method added a new Boolean column (`Ben_is_null`) showing whether each value in the `Ben` column is **NULL**. Because the `Ben` variable was created from `C6H6(GT)` after missing values were already replaced, none of its values are NULL, so all rows are marked `false`.

In [103]:
# example 3
air.check_value_missing("RH")
air.df.select("RH", "RH_is_null").show(5)

+----+----------+
|  RH|RH_is_null|
+----+----------+
|48.9|     false|
|47.7|     false|
|54.0|     false|
|60.0|     false|
|59.6|     false|
+----+----------+
only showing top 5 rows


The output indicates that the method executed successfully. The check_value_missing method added a new Boolean column (`RH_is_null`) showing whether each value in the `RH` column is `NULL`. Because missing values of the `RH` variable were already replaced, none of its values are NULL, so all rows are marked `false`.

In [106]:
# example 4
air.check_value_missing("NOx(GT)")
air.df.select("NOx(GT)", "NOx(GT)_is_null").show(5)

+-------+---------------+
|NOx(GT)|NOx(GT)_is_null|
+-------+---------------+
|    166|          false|
|    103|          false|
|    131|          false|
|    172|          false|
|    131|          false|
+-------+---------------+
only showing top 5 rows


The output indicates that the method executed successfully. The check_value_missing method added a new Boolean column (`NOx(GT)_is_null`) showing whether each value in the `NOx(GT)` column is `NULL`. Because missing values of the `NOx(GT)` variable were already replaced, none of its values are NULL, so all rows are marked `false`.


import importlib
import ST554_project2_part1
importlib.reload(ST554_project2_part1)
